In [0]:
# Inicialização de algumas constantes

catalog = "cinedata_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"


bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

# Nome das tabelas da camada bronze
tb_infos = f"{bronze_schema}.tb_movies_info"
tb_financials = f"{bronze_schema}.tb_movies_financials"
tb_metrics = f"{bronze_schema}.tb_movies_metrics"
tb_credits_and_tags = f"{bronze_schema}.tb_credits_and_tags"
tb_reviews = f"{bronze_schema}.tb_movies_reviews"

# Mapeamento dos caminhos para os arquivos brutos
path_infos = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"


In [0]:
# Criando a estrutuda basica da arquitetura
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")
print("Estrutura criada com sucesso.")




In [0]:
# Leitura dos dados brutos com tratamento explícito de aspas
def ler_csv(path):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", True)
        .csv(path)
    )

df_infos_raw = ler_csv(path_infos)
df_financials_raw = ler_csv(path_financials)
df_metrics_raw = ler_csv(path_metrics)
df_credits_and_tags_raw = ler_csv(path_credits_and_tags)
df_reviews_raw = ler_csv(path_reviews)

# A questão do uso do `append`

Como foi requisitado o uso do modo `append` para criar as tabelas da camada Bronze, surge uma preocupação relacionada à idempotência do pipeline e à duplicação de dados.

Considerando que esse pipeline seria executado automaticamente de forma recorrente, a reingestão dos mesmos arquivos adicionaria registros duplicados à camada Bronze. Além de consumir armazenamento desnecessariamente, isso aumentaria o custo computacional das etapas posteriores, que precisariam lidar com essas duplicatas.

Para mitigar esse problema, proponho calcular o hash do conteúdo de cada arquivo e utilizar essa informação para identificar arquivos já processados. Dessa forma, arquivos cujo conteúdo não sofreu alterações podem ser ignorados, enquanto arquivos com conteúdo diferente podem ser ingeridos novamente.

Essa abordagem evita a reingestão de arquivos com conteúdo idêntico, mantendo a possibilidade de incorporar novas informações quando os arquivos de origem forem atualizados. Entretanto, ela não impede duplicações entre arquivos distintos que contenham registros iguais.

Por fim, ressalto que essa estratégia foi adotada considerando a exigência de utilizar `append`. Para um cenário em que os arquivos representam um conjunto completo e atualizado dos dados de origem, o uso de `overwrite` poderia ser mais adequado, pois substituiria os dados anteriormente carregados. A escolha ideal, porém, depende da fonte e da necessidade de preservar o histórico das ingestões.

In [0]:
import hashlib
from pyspark.sql.functions import col, current_timestamp

# Tabela de controle de ingestão
tb_ingestion_control = f"{bronze_schema}.ingestion_control"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tb_ingestion_control} (
        source_path STRING,
        content_hash STRING,
        ingestion_datetime TIMESTAMP
    )
    USING DELTA
""")


def ingest_if_new(df, source_path, target_table):
    """
    Ingere um arquivo CSV para camada Bronze
    somente se seu conteúdo ainda não tiver sido ingerido.

    Calcula o hash do arquivo e verifica se já existe
    um registro correspondente na tabela de controle de ingestão.
    Caso o conteúdo seja novo, adiciona os dados à tabela de
    destino utilizando append e registra a ingestão realizada.

    Args:
        df: DataFrame contendo os dados do arquivo.
        source_path: Caminho do arquivo CSV de origem.
        target_table: Nome completo da tabela Bronze de destino.
    """


    # Calculando o hash do arquivo
    hasher = hashlib.sha256()

    with open(source_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)

    content_hash = hasher.hexdigest()

    # Verificando se esse conteúdo já foi ingerido
    already_ingested = (
        spark.table(tb_ingestion_control)
        .filter(
            (col("source_path") == source_path) &
            (col("content_hash") == content_hash)
        )
        .limit(1)
        .count() > 0
    )

    if already_ingested:
        print(f"Arquivo já ingerido, ignorando: {source_path}")
        return

    # Inserindo os dados na Bronze
    df.withColumn("ingestion_datetime", current_timestamp()) \
        .write.format("delta").mode("append") \
        .saveAsTable(target_table)

    # Registrando o processamento após a gravação
    spark.createDataFrame(
        [(source_path, content_hash)],
        ["source_path", "content_hash"]
    ) \
        .withColumn("ingestion_datetime", current_timestamp()) \
        .write.format("delta").mode("append") \
        .saveAsTable(tb_ingestion_control)

    print(f"Arquivo ingerido com sucesso: {source_path}")

In [0]:
# Criando as tabelas da camada Bronze
ingest_if_new(df_infos_raw, path_infos, tb_infos)
ingest_if_new(df_financials_raw, path_financials, tb_financials)
ingest_if_new(df_metrics_raw, path_metrics, tb_metrics)
ingest_if_new(df_credits_and_tags_raw, path_credits_and_tags, tb_credits_and_tags)
ingest_if_new(df_reviews_raw, path_reviews, tb_reviews)


In [0]:
# Verificando as tabelas criadas na camada Bronze
for table in [
    tb_infos,
    tb_financials,
    tb_metrics,
    tb_credits_and_tags,
    tb_reviews
]:
    print(f"Tabela: {table}")
    spark.table(table).show(5, truncate=False)

# Ingestão da API

Para a tabela `bronze.tb_cotacao_dolar`, optou-se por utilizar o modo `overwrite`, substituindo os dados existentes pelas cotações retornadas pela API do Banco Central a cada execução bem-sucedida.

Essa abordagem simplifica a ingestão e evita o acúmulo de registros duplicados entre execuções. Como a API pode apresentar falhas de disponibilidade, foi definida uma estratégia de contingência: caso a consulta falhe, os dados já existentes na tabela serão preservados. Se a tabela ainda não possuir registros, será utilizado um conjunto de dados mock obtido previamente da API.

Essa decisão considera que o objetivo atual é manter as cotações disponíveis para uso no processamento dos dados financeiros dos filmes, e não construir um histórico das consultas realizadas.

**Ressalva:** caso surja a necessidade de preservar o histórico das cotações ao longo do tempo, a estratégia deverá ser revista.

In [0]:
from datetime import datetime, timedelta

data_fim_padrao = datetime.today()
data_inicio_padrao = data_fim_padrao - timedelta(days=30)

dbutils.widgets.text(
    "data_inicio",
    data_inicio_padrao.strftime("%m-%d-%Y"),
    "Data inicial (MM-DD-YYYY)"
)

dbutils.widgets.text(
    "data_fim",
    data_fim_padrao.strftime("%m-%d-%Y"),
    "Data final (MM-DD-YYYY)"
)

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

In [0]:
tb_cotacao_dolar = f"{bronze_schema}.tb_cotacao_dolar"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tb_cotacao_dolar} (
        data_hora_cotacao TIMESTAMP,
        cotacao_compra DOUBLE,
        ingestion_datetime TIMESTAMP,
        source STRING
    )
    USING DELTA
""")

In [0]:
import requests

from datetime import datetime
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

# Endpoint do Banco Central
endpoint = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{data_inicio}'",
    "@dataFinalCotacao": f"'{data_fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

# Dados de contingência obtidos previamente da API
mock_rows = [
    (datetime.strptime("2026-10-13 13:04:26.866133", "%Y-%m-%d %H:%M:%S.%f"), 5.009, "mock"),
    (datetime.strptime("2026-10-14 13:05:17.833426", "%Y-%m-%d %H:%M:%S.%f"), 5.0435, "mock"),
    (datetime.strptime("2026-10-15 13:05:33.305854", "%Y-%m-%d %H:%M:%S.%f"), 5.0515, "mock"),
    (datetime.strptime("2026-10-16 13:04:14.548156", "%Y-%m-%d %H:%M:%S.%f"), 5.082, "mock"),
]

# Tenta consultar a API
api_rows = None

try:
    # Valida as datas informadas nos widgets
    inicio = datetime.strptime(data_inicio, "%m-%d-%Y")
    fim = datetime.strptime(data_fim, "%m-%d-%Y")

    if inicio > fim:
        raise ValueError("A data inicial não pode ser posterior à data final.")

    response = requests.get(
        endpoint,
        params=params,
        timeout=20
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload.get("value"), list):
        raise ValueError("Resposta da API não contém uma lista válida de cotações.")

    api_rows = []

    for item in payload["value"]:
        data_hora = datetime.strptime(
            item["dataHoraCotacao"],
            "%Y-%m-%d %H:%M:%S.%f"
        )
        cotacao = float(item["cotacaoCompra"])

        api_rows.append((data_hora, cotacao, "api"))

    print(f"Consulta à API concluída: {len(api_rows)} cotação(ões).")

except Exception as e:
    print(f"Falha na consulta à API: {e}")
    api_rows = None


# Decide quais dados persistir
if api_rows:
    # API retornou cotações: substitui os dados existentes
    rows = api_rows
    print("Utilizando os dados da API.")

elif spark.table(tb_cotacao_dolar).limit(1).count() > 0:
    # API falhou ou não retornou cotações: preserva a tabela atual
    rows = None
    print("Nenhuma cotação nova disponível. Mantendo os dados existentes.")

else:
    # Sem dados disponíveis: utiliza o mock
    rows = mock_rows
    print("Tabela vazia. Utilizando os dados mock de contingência.")


# Persiste os dados, quando necessário
if rows is not None:
    df_cotacao = spark.createDataFrame(
        rows,
        schema=[
            "data_hora_cotacao",
            "cotacao_compra",
            "source"
        ]
    ).withColumn(
        "ingestion_datetime",
        current_timestamp()
    ).select(
        "data_hora_cotacao",
        "cotacao_compra",
        "ingestion_datetime",
        "source"
    )

    if rows is not None:
        if not spark.catalog.tableExists(tb_cotacao_dolar):
            df_cotacao.write.format("delta").mode("overwrite").saveAsTable(
                tb_cotacao_dolar
            )
        else:
            (
                DeltaTable.forName(spark, tb_cotacao_dolar)
                .alias("destino")
                .merge(
                    df_cotacao.alias("origem"),
                    """
                    destino.data_hora_cotacao = origem.data_hora_cotacao
                    """
                )
                .whenMatchedUpdate(set={
                    "cotacao_compra": "origem.cotacao_compra",
                    "ingestion_datetime": "origem.ingestion_datetime",
                    "source": "origem.source"
                })
                .whenNotMatchedInsertAll()
                .execute()
            )

        print(f"Histórico atualizado: {tb_cotacao_dolar}")

In [0]:
display(
    spark.table(tb_cotacao_dolar)
    .orderBy("data_hora_cotacao")
)